In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df =pd.read_csv(r"pbl/olympic_raw_dataset.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()
df.info()


Shape: (5050, 17)
Columns: ['Country', 'Sport', 'Year', 'Athletes', 'Gold', 'Silver', 'Bronze', 'Total_Medals', 'Events_Participated', 'GDP_Index', 'Population_Millions', 'Prev_Medals', 'Host', 'Medal_per_Athlete', 'Gold_Ratio', 'Participation_Rate', 'Medals_per_Million']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              5050 non-null   object 
 1   Sport                5050 non-null   object 
 2   Year                 5050 non-null   float64
 3   Athletes             4650 non-null   float64
 4   Gold                 4648 non-null   float64
 5   Silver               4645 non-null   float64
 6   Bronze               4647 non-null   float64
 7   Total_Medals         4647 non-null   float64
 8   Events_Participated  4646 non-null   float64
 9   GDP_Index            4650 non-null   float64
 10  Population_Mill

In [5]:
# CELL 3 - CHECK MISSING VALUES BEFORE CLEANING
print("Missing values BEFORE cleaning:")
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Missing values BEFORE cleaning:
Country                  0
Sport                    0
Year                     0
Athletes               400
Gold                   402
Silver                 405
Bronze                 403
Total_Medals           403
Events_Participated    404
GDP_Index              400
Population_Millions    405
Prev_Medals            407
Host                     0
Medal_per_Athlete      775
Gold_Ratio             762
Participation_Rate     774
Medals_per_Million     778
dtype: int64
Duplicates: 50


In [6]:
# CELL 4 - DATA CLEANING
# Fix Year column
df["Year"] = pd.to_numeric(df["Year"].astype(str).str.replace(".0", "", regex=False), errors="coerce")

# Fix country name inconsistencies
df["Country"] = df["Country"].str.strip().str.title()
df["Country"] = df["Country"].replace({"Usa":"USA", "U.K.":"UK", "China":"China"})

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Fix negative values
for col in ["Gold", "Silver", "Bronze"]:
    df[col] = df[col].abs()

# Fix outliers in Athletes using IQR
Q1, Q3 = df["Athletes"].quantile(0.25), df["Athletes"].quantile(0.75)
upper  = Q3 + 1.5 * (Q3 - Q1)
df["Athletes"] = df["Athletes"].apply(lambda x: df["Athletes"].median() if pd.notna(x) and x > upper else x)

# Fill missing values with median
for col in ["Athletes","Gold","Silver","Bronze","Total_Medals",
            "Events_Participated","GDP_Index","Population_Millions","Prev_Medals"]:
    df[col].fillna(df[col].median())

# Recalculate columns
df["Total_Medals"]       = df["Gold"] + df["Silver"] + df["Bronze"]
df["Medal_per_Athlete"]  = (df["Total_Medals"] / df["Athletes"]).round(4)
df["Gold_Ratio"]         = (df["Gold"] / df["Total_Medals"].replace(0, 1)).round(4)
df["Participation_Rate"] = (df["Athletes"] / df["Events_Participated"]).round(4)
df["Medals_per_Million"] = (df["Total_Medals"] / df["Population_Millions"]).round(4)

print("Cleaned shape:", df.shape)
print("Missing values AFTER cleaning:")
print(df.isnull().sum())

Cleaned shape: (5000, 17)
Missing values AFTER cleaning:
Country                   0
Sport                     0
Year                      0
Athletes                395
Gold                    399
Silver                  400
Bronze                  400
Total_Medals           1099
Events_Participated     400
GDP_Index               397
Population_Millions     400
Prev_Medals             400
Host                      0
Medal_per_Athlete      1409
Gold_Ratio             1099
Participation_Rate      766
Medals_per_Million     1414
dtype: int64
